### FedRSClip


In [10]:
!pip install numpy torchvision 

  Obtaining dependency information for numpy from https://files.pythonhosted.org/packages/02/03/74fe2a4cb3817d94d86402f2506554130a2f01414e299b5a843e5a8a957f/numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for torchvision from https://files.pythonhosted.org/packages/fa/8a/c474fb27faba02e84dc40e0ac9ea1aa828d6d3557a378f7d0a22468bb2a3/torchvision-0.27.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for torch==2.12.1 from https://files.pythonhosted.org/packages/dc/ca/ed24783da629ff3e640ba3f70a7639e9045d3d88b93ee6bc47b8a28a1f2c/torch-2.12.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for pillow!=8.3.*,>=5.3.0 from https://files.pythonhosted.org/packages/3b/2d/ede717bc1144f63886c21fd349bb95860b0d1a21149ff16f2bb362b612b6/pillow-12.3.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for filelock from

In [18]:
!pip install scikit-learn matplotlib pandas torch transforms

  Obtaining dependency information for transforms from https://files.pythonhosted.org/packages/07/15/9b1a41ba648851defa77180fbe6c1278bae7a164a90df81ccdfa7103a4e7/transforms-0.2.1-py3-none-any.whl.metadata

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [1]:
import zipfile
import os

# Dataset ZIP file location
ZIP_PATH = "../datasets/UC_merced_land_datsets.zip"

# Extract location
EXTRACT_PATH = "../datasets"

# Create directory if it doesn't exist
os.makedirs(EXTRACT_PATH, exist_ok=True)

# Extract dataset
with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset extracted successfully!")


Dataset extracted successfully!


In [2]:
import os

for root, dirs, files in os.walk(EXTRACT_PATH):
    print(root)
    if len(files) > 0:
        print("Sample files:", files[:3])
    print("-" * 40)

../datasets
Sample files: ['UC_merced_land_datsets.zip']
----------------------------------------
../datasets/UCMerced_LandUse
Sample files: ['readme.txt.bak', 'readme.txt']
----------------------------------------
../datasets/UCMerced_LandUse/Images
----------------------------------------
../datasets/UCMerced_LandUse/Images/harbor
Sample files: ['harbor29.tif', 'harbor05.tif', 'harbor75.tif']
----------------------------------------
../datasets/UCMerced_LandUse/Images/baseballdiamond
Sample files: ['baseballdiamond15.tif', 'baseballdiamond45.tif', 'baseballdiamond34.tif']
----------------------------------------
../datasets/UCMerced_LandUse/Images/runway
Sample files: ['runway13.tif', 'runway91.tif', 'runway46.tif']
----------------------------------------
../datasets/UCMerced_LandUse/Images/freeway
Sample files: ['freeway95.tif', 'freeway91.tif', 'freeway98.tif']
----------------------------------------
../datasets/UCMerced_LandUse/Images/chaparral
Sample files: ['chaparral94.tif', 

In [3]:
from torchvision.datasets import ImageFolder
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = ImageFolder(root="../datasets/UCMerced_LandUse/Images", transform=transform)
print(dataset)

print('classes:', dataset.classes)
print('Number_of _images:', len(dataset))
print('Number_of_classes:', len(dataset.classes))



Dataset ImageFolder
    Number of datapoints: 2100
    Root location: ../datasets/UCMerced_LandUse/Images
    StandardTransform
Transform: Compose(
               Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )
classes: ['agricultural', 'airplane', 'baseballdiamond', 'beach', 'buildings', 'chaparral', 'denseresidential', 'forest', 'freeway', 'golfcourse', 'harbor', 'intersection', 'mediumresidential', 'mobilehomepark', 'overpass', 'parkinglot', 'river', 'runway', 'sparseresidential', 'storagetanks', 'tenniscourt']
Number_of _images: 2100
Number_of_classes: 21


In [6]:

import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import accuracy_score, classification_report

print("Pytorch Version : ",torch.__version__)
print("CUDA Available :",torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device: ",device)


Pytorch Version :  2.12.1+cu130
CUDA Available : False
Using Device:  cpu


In [7]:
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print("Random Seed Set: ",SEED)

NameError: name 'np' is not defined

In [5]:
#visulize Dataset Samples
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(12, 6))

for ax in axes.flatten():
    idx = np.random.randint(0, len(dataset)-1)
    img, label = dataset[idx]

    image = img.permute(1, 2, 0).numpy()
    image = (image * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
    ax.imshow(image)
    ax.set_title(dataset.classes[label])
    ax.axis('off')

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [6]:
#Train-test split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size],
                                           generator=torch.Generator().manual_seed(SEED))
print("Train Size: ",train_size)
print("Test Size: ",test_size)

NameError: name 'dataset' is not defined

In [ ]:
#load Pretrained CLIP-MODEL

MODEL_NAME = 'openai/clip-vit-base-patch32'
print("Loading CLIP Model......")

clip_model = CLIPModel.from_pretrained(MODEL_NAME)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
clip_model = clip_model.to(device)
for p in clip_model.parameters():
    p.requires_grad = False
print("Model Loaded Successfully !")

print("\n Vision Embedding Dimension:",clip_model.visual_projection.out_features)


Loading CLIP Model......


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model Loaded Successfully !

 Vision Embedding Dimension: 512


In [ ]:
#Freeze CLIP Parameters

for param in clip_model.parameters():
    param.requires_grad = False

print("Parameters Frozen Successfully !")

trainable = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)
print("Total Trainable Parameters:",trainable)

total = sum(p.numel() for p in clip_model.parameters())
print("Total Parameters:",total)

Parameters Frozen Successfully !
Total Trainable Parameters: 0
Total Parameters: 151277313


In [ ]:
import torch
import torch.nn as nn

class CLIPSceneClassifier(nn.Module):

    def __init__(self, clip_model, num_classes):
        super().__init__()

        self.clip = clip_model

        self.classifier = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, pixel_values):

        with torch.no_grad():
            vision_outputs = self.clip.vision_model(pixel_values=pixel_values)
            pooled_output = vision_outputs.pooler_output      # Tensor [B,768]
            image_features = self.clip.visual_projection(pooled_output)  # Tensor [B,512]

        logits = self.classifier(image_features)

        return logits

model = CLIPSceneClassifier(
    clip_model,
    num_classes=len(dataset.classes)
)

model = model.to(device)

print(type(model))
print(model)

<class '__main__.CLIPSceneClassifier'>
CLIPSceneClassifier(
  (clip): CLIPModel(
    (text_model): CLIPTextModel(
      (embeddings): CLIPTextEmbeddings(
        (token_embedding): Embedding(49408, 512)
        (position_embedding): Embedding(77, 512)
      )
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-11): 12 x CLIPEncoderLayer(
            (self_attn): CLIPAttention(
              (k_proj): Linear(in_features=512, out_features=512, bias=True)
              (v_proj): Linear(in_features=512, out_features=512, bias=True)
              (q_proj): Linear(in_features=512, out_features=512, bias=True)
              (out_proj): Linear(in_features=512, out_features=512, bias=True)
            )
            (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn): QuickGELUActivation()
              (fc1): Linear(in_features=512, out_features=2048, bias=True)
              (fc2): Linear(in_feature

In [ ]:
import torch

layer_outputs = {}

def hook_fn(name):
    def hook(module, input, output):

        # Handle HuggingFace outputs
        if hasattr(output, "pooler_output"):
            output = output.pooler_output

        elif isinstance(output, tuple):
            output = output[0]

        if torch.is_tensor(output):
            layer_outputs[name] = output.detach().cpu()

    return hook


# Patch Embedding
model.clip.vision_model.embeddings.register_forward_hook(
    hook_fn("Patch Embedding")
)

# 12 Transformer Blocks
for i, block in enumerate(model.clip.vision_model.encoder.layers):
    block.register_forward_hook(
        hook_fn(f"Transformer Block {i+1}")
    )

# LayerNorm
model.clip.vision_model.post_layernorm.register_forward_hook(
    hook_fn("Post LayerNorm")
)

# Projection Layer
model.clip.visual_projection.register_forward_hook(
    hook_fn("Visual Projection")
)

# Classifier Layers
model.classifier[0].register_forward_hook(
    hook_fn("Classifier FC1")
)

model.classifier[1].register_forward_hook(
    hook_fn("ReLU")
)

model.classifier[2].register_forward_hook(
    hook_fn("Dropout")
)

model.classifier[3].register_forward_hook(
    hook_fn("Classifier FC2")
)

print("Hooks Registered Successfully!")

Hooks Registered Successfully!


In [ ]:
#forward pass

images, labels = next(iter(train_loader))

images = images.to(device)

model.eval()

with torch.no_grad():
    outputs = model(images)

In [ ]:
#Layer Information

print("="*80)
print("CLIP Vision Transformer Layer Outputs")
print("="*80)

for name, value in layer_outputs.items():

    print(f"\n{name}")

    print("-"*60)

    print("Shape :", tuple(value.shape))

    print("Mean  :", value.mean().item())

    print("Std   :", value.std().item())

    print("Min   :", value.min().item())

    print("Max   :", value.max().item())

CLIP Vision Transformer Layer Outputs

Patch Embedding
------------------------------------------------------------
Shape : (32, 50, 768)
Mean  : -0.057794149965047836
Std   : 1.2770462036132812
Min   : -26.51561164855957
Max   : 27.480762481689453

Transformer Block 1
------------------------------------------------------------
Shape : (32, 50, 768)
Mean  : 0.005289499182254076
Std   : 0.17510844767093658
Min   : -5.409224033355713
Max   : 1.9971330165863037

Transformer Block 2
------------------------------------------------------------
Shape : (32, 50, 768)
Mean  : 0.008255094289779663
Std   : 0.1674339473247528
Min   : -6.019175052642822
Max   : 1.9196739196777344

Transformer Block 3
------------------------------------------------------------
Shape : (32, 50, 768)
Mean  : 0.011417004279792309
Std   : 0.14995920658111572
Min   : -6.049618244171143
Max   : 1.9411380290985107

Transformer Block 4
------------------------------------------------------------
Shape : (32, 50, 768)
Mea

In [ ]:
#Create Datasets Batches
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,pin_memory=True)

print("Train Batches size", len(train_loader))
print("Test Batches size", len(test_loader))
print(train_loader.dataset)
print(test_loader.dataset)

Train Batches size 53
Test Batches size 14


In [ ]:
#loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("Loss Functon", criterion)
print("Optimizer", optimizer)

Loss Functon CrossEntropyLoss()
Optimizer Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)


In [ ]:
#Tranning Function

def train_one_epoch(model, loader, criterion, optimizer):
  model.train()
  runningloss = 0.0
  correct = 0
  total = 0

  for images, labels in loader:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    outputs = model(images) #forward pass
    loss = criterion(outputs, labels) #loss calculation
    loss.backward() #backword pass
    optimizer.step()  #update weights

    runningloss += loss.item()
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

  epoch_loss = runningloss / len(loader)
  epoch_acc = 100 * correct / total
  return epoch_loss, epoch_acc

In [ ]:
#validation function
def evalute_one_epoch(model, loader, criterion):
  model.eval()
  runningloss = 0.0
  correct = 0
  total = 0
  all_preds = []
  all_labels = []

  with torch.no_grad():
    for images, labels in loader:
      images = images.to(device)
      labels = labels.to(device)

      outputs = model(images)
      loss = criterion(outputs, labels)

      runningloss += loss.item()
      _, predicted = torch.max(outputs.data, 1)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()

      all_preds.extend(predicted.cpu().numpy())
      all_labels.extend(labels.cpu().numpy())

  epoch_loss = runningloss / len(loader)
  epoch_acc = 100 * correct / total
  return epoch_loss, epoch_acc, all_preds, all_labels

In [ ]:
images, labels = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():

    vision_outputs = model.clip.vision_model(
        pixel_values=images
    )

print(type(vision_outputs))

print(type(vision_outputs.pooler_output))

print(vision_outputs.pooler_output.shape)

features = model.clip.visual_projection(
    vision_outputs.pooler_output
)

print(features.shape)


<class 'transformers.modeling_outputs.BaseModelOutputWithPooling'>
<class 'torch.Tensor'>
torch.Size([32, 768])
torch.Size([32, 512])


In [ ]:
import inspect
print(inspect.getsource(model.forward))

    def forward(self, pixel_values):

        with torch.no_grad():
            vision_outputs = self.clip.vision_model(pixel_values=pixel_values)
            pooled_output = vision_outputs.pooler_output      # Tensor [B,768]
            image_features = self.clip.visual_projection(pooled_output)  # Tensor [B,512]

        logits = self.classifier(image_features)

        return logits



In [ ]:

# Training Loop

EPOCHS = 10

train_losses = []
test_losses = []

train_accs = []
test_accs = []

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    test_loss, test_acc, preds, labels = evalute_one_epoch(
        model,
        test_loader,
        criterion
    )

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% | "
        f"Test Loss: {test_loss:.4f} "
        f"Test Acc: {test_acc:.2f}%"
    )

Epoch [1/10] Train Loss: 2.9025 Train Acc: 22.50% | Test Loss: 2.7320 Test Acc: 62.86%
Epoch [2/10] Train Loss: 2.5133 Train Acc: 63.81% | Test Loss: 2.3180 Test Acc: 75.24%
Epoch [3/10] Train Loss: 2.0470 Train Acc: 77.44% | Test Loss: 1.8508 Test Acc: 79.76%
Epoch [4/10] Train Loss: 1.6044 Train Acc: 81.13% | Test Loss: 1.4353 Test Acc: 84.52%
Epoch [5/10] Train Loss: 1.2551 Train Acc: 83.93% | Test Loss: 1.1292 Test Acc: 87.62%
Epoch [6/10] Train Loss: 0.9936 Train Acc: 86.49% | Test Loss: 0.9099 Test Acc: 89.76%
Epoch [7/10] Train Loss: 0.8172 Train Acc: 86.79% | Test Loss: 0.7567 Test Acc: 89.76%
Epoch [8/10] Train Loss: 0.6764 Train Acc: 89.70% | Test Loss: 0.6521 Test Acc: 90.71%
Epoch [9/10] Train Loss: 0.5944 Train Acc: 90.06% | Test Loss: 0.5725 Test Acc: 90.71%
Epoch [10/10] Train Loss: 0.5206 Train Acc: 90.71% | Test Loss: 0.5144 Test Acc: 90.95%
